In [17]:
import warnings
warnings.filterwarnings("ignore")

# Data Preparation

In [1]:
import nltk
from nltk.corpus import wordnet as wn
import asyncio
from googletrans import Translator
from itertools import islice
import json

In [2]:
translator = Translator()
text = "Hello, i am using google cloud"

result = await translator.translate(text, dest="hi")
print(result.text)

नमस्ते, मैं गूगल क्लाउड का उपयोग कर रहा हूं


In [3]:
nltk.download("wordnet")
nltk.download("omw-1.4")

english_sentences = set()

for synset in wn.all_synsets():
    for ex in synset.examples():
        if 5 < len(ex.split()) < 25:
            english_sentences.add(ex)

english_sentences_1 = list(english_sentences)
print("Total English sentences:", len(english_sentences_1))


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Total English sentences: 23582


In [4]:
with open("english_sentences.txt", "w", encoding="utf-8") as f:
    for s in english_sentences:
        f.write(s + "\n")

In [5]:
translator = Translator()

async def safe_batch_translate(
    sentences,
    dest="hi",
    batch_size=50,     # DO NOT increase beyond 100
    delay=1.0,         # seconds
    max_retries=3
):
    translated_texts = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        for attempt in range(max_retries):
            try:
                results = await translator.translate(batch, dest=dest)
                translated_texts.extend([r.text for r in results])
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"Failed batch {i}-{i+batch_size}: {e}")
                    translated_texts.extend([""] * len(batch))
                else:
                    await asyncio.sleep(2)  # wait before retry

        await asyncio.sleep(delay)  # polite delay

        if i % 500 == 0:
            print(f"Translated {i}/{len(sentences)}")

    return translated_texts

In [ ]:
english_subset = list(islice(english_sentences, 15000, 20000))

hindi_texts = await safe_batch_translate(
    english_subset,
    dest="hi",
    batch_size=50,
    delay=1.0
)

print(hindi_texts)

✅ Translated 0/5000


In [6]:
data = [{"english": en, "hindi": hi} for en, hi in zip(english_subset, hindi_texts)]

with open("english_hindi_translations.json", "a", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Converted JSON → JSONL")

✅ Converted JSON → JSONL


In [ ]:
with open("english_hindi_test_translations.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("english_hindi_test_translations.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("✅ Converted JSON → JSONL")

In [2]:
train_data = []
with open("english_hindi.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))

print("Original train size:", len(train_data))


test_data = []
with open("english_hindi_test_translations.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line))

print("Test size:", len(test_data))

Original train size: 15000
Test size: 5000


In [3]:
train_data.extend(test_data)
print("Combined data size:", len(train_data))

Combined data size: 20000


In [4]:
with open("english_hindi_full_corpus.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved combined corpus as english_hindi_full_corpus.jsonl")

Saved combined corpus as english_hindi_full_corpus.jsonl


# Model Training

In [3]:
import torch
import random
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
import torch.nn.functional as F

In [4]:
SEED = 42
BATCH_SIZE = 16
EPOCHS = 3
TEMPERATURE = 0.05

In [5]:
sentences = []

with open("english_hindi_full_corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        sentences.append(obj["hindi"])

print("Total Hindi sentences:", len(sentences))

Total Hindi sentences: 20000


In [6]:
MODEL_NAME = "bert-base-multilingual-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(105879, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [7]:
def encode_sentences(sentences):
    inputs = tokenizer(
        list(sentences),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1)

In [8]:
def contrastive_loss(z1, z2, temperature=TEMPERATURE):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(z1.size(0)).to(z1.device)

    loss_1 = F.cross_entropy(logits, labels)
    loss_2 = F.cross_entropy(logits.T, labels)

    return (loss_1 + loss_2) / 2

In [10]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

loader = DataLoader(sentences, batch_size=BATCH_SIZE, shuffle=True)
optimizer = AdamW(model.parameters(), lr=LR)

model.train()

for epoch in range(EPOCHS):
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        z1 = encode_sentences(batch)
        z2 = encode_sentences(batch)   # same sentences, dropout diff

        loss = contrastive_loss(z1, z2)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

Epoch 1 | Loss: 0.0009
Epoch 2 | Loss: 0.0006
Epoch 3 | Loss: 0.0003


In [11]:
model.eval()
for param in model.parameters():
    param.requires_grad = False

In [12]:
import unicodedata
import re
from collections import defaultdict

def normalize_hindi_word(word):
    word = unicodedata.normalize("NFC", word)
    word = re.sub(r"[।,!?]", "", word)
    return word.strip()


def extract_word_embeddings(sentence):
    encoding = tokenizer(
        sentence,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True
    )

    offsets = encoding["offset_mapping"][0].tolist()
    input_ids = encoding["input_ids"].to(device)

    with torch.no_grad():
        outputs = model(input_ids)
        hidden = outputs.last_hidden_state[0]

    words = {}
    current_word = ""
    current_embs = []
    prev_end = None

    for (start, end), emb in zip(offsets, hidden):
        if start == end:
            continue

        if prev_end is None or start != prev_end:
            if current_word:
                norm = normalize_hindi_word(current_word)
                if norm:
                    words[norm] = torch.stack(current_embs).mean(0).cpu().numpy()
            current_word = sentence[start:end]
            current_embs = [emb]
        else:
            current_word += sentence[start:end]
            current_embs.append(emb)

        prev_end = end

    if current_word:
        norm = normalize_hindi_word(current_word)
        if norm:
            words[norm] = torch.stack(current_embs).mean(0).cpu().numpy()

    return words

In [13]:
token_contexts = defaultdict(list)

for idx, sent in enumerate(sentences):
    word_embs = extract_word_embeddings(sent)
    for word, emb in word_embs.items():
        token_contexts[word].append((idx, emb))

print("Total unique words:", len(token_contexts))

Total unique words: 12565


In [18]:
import numpy as np
from sklearn.cluster import KMeans
import os
os.environ["OMP_NUM_THREADS"] = "1"

word_senses = {}

for word, contexts in token_contexts.items():

    if len(contexts) < 2:
        continue

    sent_ids, embs = zip(*contexts)
    X = np.vstack(embs)

    # remove duplicate embeddings
    X_unique = np.unique(X, axis=0)

    if len(X_unique) < 2:
        continue

    k = min(3, len(X_unique))

    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_unique)

    word_senses[word] = {
        "labels": labels,
        "centroids": kmeans.cluster_centers_,
        "sent_ids": sent_ids[:len(X_unique)]
    }

print("Total words with induced senses:", len(word_senses))

Total words with induced senses: 6814


In [19]:
import pickle

with open("sense_inventory.pkl", "wb") as f:
    pickle.dump(word_senses, f)

torch.save(model.state_dict(), "trained_mbert.pt")
1
print("Model and sense inventory saved.")

Model and sense inventory saved.


# Model Testing

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")
model = AutoModel.from_pretrained("bert-base-multilingual-uncased").to(device)
model.eval()

with open("sense_inventory.pkl", "rb") as f:
    sense_inventory = pickle.load(f)

# move centroids to GPU
for w in sense_inventory:
    sense_inventory[w]["centroids"] = torch.tensor(
        sense_inventory[w]["centroids"],
        dtype=torch.float32,
        device=device
    )

print("Model and sense inventory loaded")

Model and sense inventory loaded


In [21]:
train_data = []
with open("english_hindi_full_corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))

print("Corpus size:", len(train_data))

Corpus size: 20000


In [41]:
def encode_sentences(sentences):
    inputs = tokenizer(
        list(sentences),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1)

In [42]:
def predict_sense(word: str, sentence: str):
    if word not in sense_inventory:
        return None

    sentence_embedding = encode_sentence(sentence)

    centroids = sense_inventory[word]["centroids"]

    sims = F.cosine_similarity(
        sentence_embedding.unsqueeze(0),
        centroids,
        dim=1
    )

    return torch.argmax(sims).item()


In [43]:
def get_english_examples(word, sense_id, top_k=3):
    sent_ids = sense_inventory[word]["sent_ids"]
    labels = sense_inventory[word]["labels"]

    examples = []
    for sid, lbl in zip(sent_ids, labels):
        if lbl == sense_id and sid < len(train_data):
            examples.append(train_data[sid]["english"])
        if len(examples) == top_k:
            break

    return examples

In [44]:
sentence_hi = input("Enter Hindi sentence: ").strip()
target_word = input("Enter target Hindi word: ").strip()

Enter Hindi sentence:  उसने अपने मित्र को पत्र लिखा
Enter target Hindi word:  पत्र 


In [45]:
if target_word not in sentence_hi.split():
    print("Target word not found in sentence")

elif target_word not in sense_inventory:
    print("Target word not in sense inventory")

else:
    sense_id = predict_sense(target_word, sentence_hi)

    if sense_id is None:
        print("Could not predict sense")

    else:
        examples = get_english_examples(target_word, sense_id)

        print("\nWord Sense Disambiguation Result")
        print("Hindi word:", target_word)
        print("Sentence:", sentence_hi)
        print("Predicted meaning (English examples):")

        if not examples:
            print("No examples found for this sense")
        else:
            for ex in examples:
                print("•", ex)


Word Sense Disambiguation Result
Hindi word: पत्र
Sentence: उसने अपने मित्र को पत्र लिखा
Predicted meaning (English examples):
• They scraped a letter into the stone
• This letter is being circulated among the faculty
• his grandmother taught him his letters


In [46]:
sense_map = {
    "पत्र": {
        0: "letter",
        1: "newspaper",
        2: "leaf",
        3: "exam paper",
        4: "official document"
    },
    "बैंक": {
        0: "financial institution",
        1: "river bank"
    },
    "जड़": {
        0: "plant root",
        1: "cause",
        2: "mathematical root"
    },
    "धारा": {
        0: "stream",
        1: "law section",
        2: "electric current"
    },
    "मुख": {
        0: "face",
        1: "mouth",
        2: "entrance",
        3: "river mouth"
    },
    "कलम": {
        0: "pen",
        1: "writing style",
        2: "writing skill"
    },
    "चाल": {
        0: "walk",
        1: "move",
        2: "trick",
        3: "strategy"
    },
    "हल": {
        0: "solution",
        1: "plough"
    },
    "तीर": {
        0: "arrow",
        1: "river bank"
    },
    "रूप": {
        0: "appearance",
        1: "form"
    },
    "खेल": {
        0: "sport",
        1: "game"
    },
    "किताब": {
        0: "book"
    },
    "मौसम": {
        0: "weather"
    },
    "घड़ी": {
        0: "clock"
    },
    "दरवाजा": {
        0: "door"
    },
    "रास्ता": {
        0: "road"
    },
    "सच": {
        0: "truth"
    },
    "स्कूल": {
        0: "school"
    },
    "चाय": {
        0: "tea"
    },
    "छात्र": {
        0: "student"
    },
    "फिल्म": {
        0: "movie"
    },
    "पक्षी": {
        0: "bird"
    },
    "गर्मी": {
        0: "heat"
    },
    "गलती": {
        0: "mistake"
    },
    "काम": {
        0: "work"
    },
    "घर": {
        0: "home"
    },
    "पानी": {
        0: "water"
    },
    "मैदान": {
        0: "field"
    },
    "गेंद": {
        0: "ball"
    },
    "दुकान": {
        0: "shop"
    },
    "नदी": {
        0: "river"
    },
    "पेड़": {
        0: "tree"
    },
    "फल": {
        0: "fruit",
        1: "result"
    },
    "समय": {
        0: "time"
    },
    "कुर्सी": {
        0: "chair",
        1: "position"
    },
    "हाथ": {
        0: "hand",
        1: "control"
    },
    "आंख": {
        0: "eye",
        1: "vision"
    },
    "दिल": {
        0: "heart",
        1: "emotion"
    },
    "रोशनी": {
        0: "light"
    },
    "आवाज": {
        0: "sound",
        1: "voice"
    }
}

In [50]:
import json
# load gold data
with open("gold_senses_200.json", "r", encoding="utf-8") as f:
    gold_data = json.load(f)

correct = 0
total = 0
skipped = 0

for item in gold_data:
    sentence = item["sentence"]
    word = item["target_word"]
    gold = item["gold_sense"]

    pred_sense_id = predict_sense(word, sentence)

    # skip if model cannot predict
    if pred_sense_id is None:
        skipped += 1
        continue

    # skip if mapping missing
    if word not in sense_map or pred_sense_id not in sense_map[word]:
        skipped += 1
        continue

    pred = sense_map[word][pred_sense_id]

    total += 1
    if pred == gold:
        correct += 1

accuracy = correct / total * 100 if total > 0 else 0.0

print(f"Evaluated: {total}")
print(f"Skipped: {skipped}")
print(f"Accuracy: {accuracy:.2f}%")

Evaluated: 90
Skipped: 110
Accuracy: 57.78%
